# Stock Intelligence Platform — GPU training

Parent → child transfer learning → price-space evaluation → champion artifact → prediction export.

**Runtime:** `Runtime → Change runtime type → GPU (T4)` before running anything.

This notebook runs repository code. It does not re-implement the training loop.

## 0. On your laptop (before Colab)

Build the upload ZIP with Python — **do not use** PowerShell `Compress-Archive` (broken `\\` paths on Colab).

```powershell
cd "C:\Harish\AI projects\Stock-Agent-Ops-Cursor\Stock Intelligence Platform"
.\.venv\Scripts\python.exe scripts\make_colab_bundle.py
```

That creates `stock-intelligence-colab.zip` containing `src/`, tests, and `feature_store/data/features.parquet`.

Then upload **this notebook** to Colab and run cells top to bottom.

## 1. Upload and extract the project ZIP

Run the next cell, choose `stock-intelligence-colab.zip`, and wait until it prints `src? True` and `parquet? True`.

In [ ]:
from pathlib import Path
import os
import shutil
import zipfile

from google.colab import files

PROJECT_DIR = Path("/content/stock-intelligence-platform")


def extract_zip_posix(archive: Path, destination: Path) -> None:
    """Extract ZIP members, converting Windows backslashes into real directories."""
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        for info in zf.infolist():
            member = info.filename.replace("\\", "/")
            if not member or member.endswith("/"):
                continue
            target = destination / member
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as source, target.open("wb") as handle:
                shutil.copyfileobj(source, handle)


uploaded = files.upload()
archive = Path("/content") / next(iter(uploaded))
extract_zip_posix(archive, PROJECT_DIR)
os.chdir(PROJECT_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("cwd:", Path.cwd())
print("src?", Path("src").is_dir())
print("tests?", Path("tests").is_dir())
print("parquet?", Path("feature_store/data/features.parquet").is_file())
assert Path("src").is_dir(), "src/ missing after extract"
assert Path("feature_store/data/features.parquet").is_file(), "features.parquet missing after extract"

## 2. Install dependencies and verify GPU

In [ ]:
%pip install -q yfinance pyarrow scikit-learn joblib mlflow matplotlib pytest
%pip install -q -e "/content/stock-intelligence-platform" --no-deps

In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Select Runtime → Change runtime type → GPU, then Runtime → Restart session"
print("GPU:", torch.cuda.get_device_name(0))
print("cwd:", Path.cwd())
print("top-level:", sorted(p.name for p in Path(".").iterdir() if not p.name.startswith(".")))

test_files = [
    Path("tests/test_data_pipeline.py"),
    Path("tests/test_model_core.py"),
    Path("tests/test_pipelines.py"),
]
present = [path for path in test_files if path.exists()]
if present:
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", *[str(path) for path in present]],
        check=False,
    )
    print("pytest exit code:", result.returncode)
else:
    print("tests/ missing — skipping pytest; training can continue")

## 3. Verify the uploaded parquet

Training uses this file only (`source="feature-store"`). No yfinance rebuild for training.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from src.config import Config
from src.pipelines.data_pipeline import dataset_report, load_features
from src.pipelines.training_pipeline import train_child, train_parent

cfg = Config()
feature_path = Path("feature_store/data/features.parquet")
assert feature_path.exists(), f"Missing uploaded parquet: {feature_path}"

print(json.dumps(dataset_report(), indent=2, default=str))
for ticker in [cfg.parent_ticker, "NVDA"]:
    frame = load_features(ticker, cfg=cfg)
    print(ticker, frame.shape, frame["date"].min(), "->", frame["date"].max())
    display(frame.head(3))

## 4. Inspect the exact PyTorch code that will run

Printed from `src/model/definition.py` and `src/model/training.py`:

- LSTM → Dropout → Linear horizon head
- Loss: `MSELoss`
- Optimizer: `Adam`
- Scheduler: `ReduceLROnPlateau`
- Gradient clip: 1.0
- Early stopping + restore best validation weights

In [ ]:
import inspect

from src.model.definition import LSTMForecaster
from src.model.training import train_model

print(inspect.getsource(LSTMForecaster))
print(inspect.getsource(train_model))

In [ ]:
PARENT_EPOCHS = 50
CHILD_EPOCHS = 30
CHILD_TICKERS = ["NVDA"]
MINIMUM_CHILD_IMPROVEMENT = 0.01

model_preview = LSTMForecaster(
    input_size=cfg.input_size,
    hidden_size=cfg.hidden_size,
    layers=cfg.num_layers,
    horizon=cfg.pred_len,
    dropout=cfg.dropout,
)
training_spec = {
    "device": cfg.device,
    "features": cfg.features,
    "lookback": cfg.context_len,
    "horizon": cfg.pred_len,
    "batch_size": cfg.batch_size,
    "parent_epochs_max": PARENT_EPOCHS,
    "child_epochs_max": CHILD_EPOCHS,
    "parent_learning_rate": cfg.learning_rate,
    "child_strategy": cfg.transfer_strategy,
    "child_learning_rate": (
        cfg.learning_rate if cfg.transfer_strategy == "freeze" else cfg.fine_tune_lr
    ),
    "loss": "MSELoss",
    "optimizer": "Adam",
    "scheduler": "ReduceLROnPlateau(factor=0.5, patience=3)",
    "early_stopping_patience": 10,
    "gradient_clip_norm": 1.0,
    "seed": cfg.seed,
}
print(json.dumps(training_spec, indent=2))
print(model_preview)

## 5. Train parent (`^GSPC`)

In [ ]:
parent_summary = train_parent(epochs=PARENT_EPOCHS, source="feature-store")
print(json.dumps({k: v for k, v in parent_summary.items() if k != "history"}, indent=2, default=str))

In [ ]:
parent_history = pd.DataFrame(parent_summary["history"])
parent_history.index = parent_history.index + 1
parent_history.index.name = "epoch"
parent_history[["train_loss", "validation_loss"]].plot(
    title="Parent loss", grid=True, figsize=(9, 4)
)
display(parent_history.tail())
print("best epoch:", parent_summary["best_epoch"], "best val loss:", parent_summary["best_loss"])

## 6. Fine-tune child (`NVDA`) and select champion

In [ ]:
child_summaries = {}
for ticker in CHILD_TICKERS:
    child_summaries[ticker] = train_child(
        ticker,
        epochs=CHILD_EPOCHS,
        child_improvement=MINIMUM_CHILD_IMPROVEMENT,
        source="feature-store",
    )

for ticker, summary in child_summaries.items():
    printable = {k: v for k, v in summary.items() if k != "history"}
    print(json.dumps(printable, indent=2, default=str))

In [ ]:
for ticker, summary in child_summaries.items():
    history = pd.DataFrame(summary["history"])
    history.index = history.index + 1
    history.index.name = "epoch"
    history[["train_loss", "validation_loss"]].plot(
        title=f"{ticker} child loss", grid=True, figsize=(9, 4)
    )
    display(history.tail())
    print(
        ticker,
        "best epoch:", summary["best_epoch"],
        "best val loss:", summary["best_loss"],
        "champion:", summary["champion"],
        "promoted:", summary["promoted"],
    )

In [ ]:
rows = [
    {"ticker": cfg.parent_ticker, "candidate": candidate, **metrics}
    for candidate, metrics in parent_summary["metrics"].items()
]
for ticker, summary in child_summaries.items():
    rows.extend(
        [
            {"ticker": ticker, "candidate": "parent", **summary["parent_metrics"]["model"]},
            {"ticker": ticker, "candidate": "child", **summary["child_metrics"]["model"]},
            {
                "ticker": ticker,
                "candidate": "persistence",
                **summary["child_metrics"]["persistence"],
            },
        ]
    )

metrics = pd.DataFrame(rows)
display(metrics.sort_values(["ticker", "mae"]))

## 7. Save predictions and download artifacts

Child artifacts are written only when the child wins the champion gate.

In [ ]:
from src.pipelines.inference_pipeline import predict_child, predict_parent

prediction_dir = Path("outputs/predictions")
prediction_dir.mkdir(parents=True, exist_ok=True)

for ticker in CHILD_TICKERS:
    champion = child_summaries[ticker]["champion"]
    prediction = (
        predict_child(ticker, cfg.pred_len)
        if champion == "child"
        else predict_parent(ticker, cfg.pred_len)
    )
    if champion == "persistence":
        for point in prediction["predictions"]:
            point["close"] = prediction["last_close"]
            point["value"] = prediction["last_close"]
    prediction["selected_candidate"] = champion
    destination = prediction_dir / f"{ticker}.json"
    destination.write_text(json.dumps(prediction, indent=2), encoding="utf-8")
    print(destination, champion)
    display(pd.DataFrame(prediction["predictions"]))

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/sip_colab_outputs", "zip", PROJECT_DIR / "outputs")
print("download:", archive)
print("contents:", sorted(p.as_posix() for p in (PROJECT_DIR / "outputs").rglob("*") if p.is_file()))
files.download(archive)

## 8. Back in Cursor

```powershell
cd "C:\Harish\AI projects\Stock-Agent-Ops-Cursor\Stock Intelligence Platform"
Expand-Archive -Path sip_colab_outputs.zip -DestinationPath outputs -Force
Get-ChildItem outputs -Recurse
```

Then serve locally — do **not** retrain on the laptop.